# Notebook 18a: Pathway NN Data Preparation

## Purpose
Generate degree-binned training data from the original Hetionet graph for pathway count prediction.

## Strategy
Instead of training on millions of individual node pairs, we:
1. Bin nodes by degree (10x10 bins for source/target)
2. Compute pathway counts per bin
3. Compute intermediate node degree signatures per bin
4. Create ~100 training samples (degree bin combinations)

This reduces memory by 1000x while capturing essential degree structure.

## Inputs
- Original Hetionet graph (data/hetmat/)
- Metapath definition (edge1, edge2)

## Outputs
- results/pathway_nn/training_data/{metapath}_training_data.csv
  - Columns: source_bin, target_bin, inter_sig_0...inter_sig_99, pathway_count_mean, pathway_count_std

## Parameters (Papermill)
- metapath: str (e.g., 'CbGpPW')
- edge1_type: str (e.g., 'CbG')
- edge2_type: str (e.g., 'GpPW')
- n_degree_bins: int = 10
- n_inter_bins: int = 10

In [ ]:
# Papermill parameters
metapath = 'CbGpPW'
edge1_type = 'CbG'
edge2_type = 'GpPW'
n_degree_bins = 10
n_inter_bins = 10
random_seed = 42

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
from pathlib import Path
import sys

# Add src to path
repo_dir = Path.cwd().parent
sys.path.insert(0, str(repo_dir))

from src.intermediate_signatures import (
    compute_intermediate_signature,
    create_degree_bins,
    extract_training_features,
    get_signature_stats
)

print(f"Repository: {repo_dir}")
print(f"Metapath: {metapath}")
print(f"Edge1: {edge1_type}, Edge2: {edge2_type}")

In [ ]:
# Create output directory
output_dir = repo_dir / 'results' / 'pathway_nn' / 'training_data'
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {output_dir}")

## Load Original Graph

In [ ]:
# Load edge matrices from original graph
data_dir = repo_dir / 'data'

# Try primary location first (local development)
edge1_file = data_dir / 'edges' / f'{edge1_type}.sparse.npz'
if not edge1_file.exists():
    # Fallback to hetmat location in permutations/000 (HPC/canonical path)
    edge1_file = data_dir / 'permutations' / '000.hetmat' / 'edges' / f'{edge1_type}.sparse.npz'

edge2_file = data_dir / 'edges' / f'{edge2_type}.sparse.npz'
if not edge2_file.exists():
    # Fallback to hetmat location in permutations/000 (HPC/canonical path)
    edge2_file = data_dir / 'permutations' / '000.hetmat' / 'edges' / f'{edge2_type}.sparse.npz'

if not edge1_file.exists():
    raise FileNotFoundError(f"Edge file not found: {edge1_file}")
if not edge2_file.exists():
    raise FileNotFoundError(f"Edge file not found: {edge2_file}")

edge1_matrix = sp.load_npz(str(edge1_file))
edge2_matrix = sp.load_npz(str(edge2_file))

print(f"Loaded edge files:")
print(f"  Edge1: {edge1_file}")
print(f"  Edge2: {edge2_file}")
print(f"Edge1 ({edge1_type}): {edge1_matrix.shape}, {edge1_matrix.nnz:,} edges, dtype={edge1_matrix.dtype}")
print(f"Edge2 ({edge2_type}): {edge2_matrix.shape}, {edge2_matrix.nnz:,} edges, dtype={edge2_matrix.dtype}")

# CRITICAL: Convert boolean matrices to numeric BEFORE computing pathways
# Boolean @ Boolean gives reachability (True/False), NOT path counts!
if edge1_matrix.dtype == bool or edge1_matrix.dtype == np.bool_:
    print(f"Converting Edge1 from boolean to int32 for path counting")
    edge1_matrix = edge1_matrix.astype(np.int32)
if edge2_matrix.dtype == bool or edge2_matrix.dtype == np.bool_:
    print(f"Converting Edge2 from boolean to int32 for path counting")
    edge2_matrix = edge2_matrix.astype(np.int32)

print(f"After conversion:")
print(f"  Edge1 dtype: {edge1_matrix.dtype}")
print(f"  Edge2 dtype: {edge2_matrix.dtype}")

## Compute Node Degrees

In [ ]:
# Compute degrees
source_degrees = np.asarray(edge1_matrix.sum(axis=1)).ravel()
target_degrees = np.asarray(edge2_matrix.sum(axis=1)).ravel()

print(f"Source nodes: {len(source_degrees)}")
print(f"  Degree range: {source_degrees[source_degrees>0].min():.0f} - {source_degrees.max():.0f}")
print(f"  Mean degree: {source_degrees[source_degrees>0].mean():.1f}")

print(f"\nTarget nodes: {len(target_degrees)}")
print(f"  Degree range: {target_degrees[target_degrees>0].min():.0f} - {target_degrees.max():.0f}")
print(f"  Mean degree: {target_degrees[target_degrees>0].mean():.1f}")

## Create Degree Bins

In [ ]:
# Create quantile-based degree bins
source_bins = create_degree_bins(source_degrees, n_degree_bins)
target_bins = create_degree_bins(target_degrees, n_degree_bins)

print(f"Source bins ({len(source_bins)-1} bins): {source_bins}")
print(f"Target bins ({len(target_bins)-1} bins): {target_bins}")
print(f"\nTotal degree bin combinations: {(len(source_bins)-1) * (len(target_bins)-1)}")

## Compute Intermediate Signatures

In [ ]:
print("Computing intermediate node degree signatures...")
print(f"This creates a {n_inter_bins}x{n_inter_bins} histogram of (in_degree, out_degree)")
print(f"for intermediate nodes in each source-target degree bin pair.\n")

signatures = compute_intermediate_signature(
    edge1_matrix=edge1_matrix,
    edge2_matrix=edge2_matrix,
    source_degrees=source_degrees,
    target_degrees=target_degrees,
    source_bins=source_bins,
    target_bins=target_bins,
    n_intermediate_bins=n_inter_bins
)

print(f"✓ Computed signatures for {len(signatures)} degree bin pairs")

# Print statistics
stats = get_signature_stats(signatures)
print(f"\nSignature Statistics:")
for key, value in stats.items():
    print(f"  {key}: {value}")

## Compute Pathway Counts per Bin

In [ ]:
print("Computing pathway counts for each degree bin pair...\n")

# Compute pathway matrix (edge matrices already converted to int32 in Cell 5)
pathway_matrix = edge1_matrix @ edge2_matrix

print(f"Pathway matrix: {pathway_matrix.shape}, {pathway_matrix.nnz:,} non-zero pathways")
print(f"Pathway matrix dtype: {pathway_matrix.dtype}")
print(f"Value range: {pathway_matrix.data.min()} - {pathway_matrix.data.max()}")

# Convert to COO for efficient access
pathway_coo = pathway_matrix.tocoo()
pathway_dict = {(i, j): v for i, j, v in zip(pathway_coo.row, pathway_coo.col, pathway_coo.data)}

print(f"Total pathway entries: {len(pathway_dict):,}")

In [ ]:
# Assign source and target nodes to bins
from src.intermediate_signatures import assign_to_bins

source_bin_assignments = assign_to_bins(source_degrees, source_bins)
target_bin_assignments = assign_to_bins(target_degrees, target_bins)

# Aggregate pathway counts by bin
bin_pathway_counts = {}  # (src_bin, tgt_bin) -> list of counts

for (i, j), count in pathway_dict.items():
    src_bin = source_bin_assignments[i]
    tgt_bin = target_bin_assignments[j]
    
    key = (src_bin, tgt_bin)
    if key not in bin_pathway_counts:
        bin_pathway_counts[key] = []
    bin_pathway_counts[key].append(count)

print(f"Pathway counts aggregated for {len(bin_pathway_counts)} bins")

## Create Training Dataset

In [ ]:
print("Creating training dataset...\n")

# Extract normalized signature features
X_signatures, bin_pairs = extract_training_features(signatures, normalize=True)

print(f"Signature features: {X_signatures.shape}")
print(f"Bin pairs: {bin_pairs.shape}")

# Build training dataframe
training_data = []

for idx, (src_bin, tgt_bin) in enumerate(bin_pairs):
    # Get signature
    sig_features = X_signatures[idx]
    
    # Get pathway count statistics
    counts = bin_pathway_counts.get((src_bin, tgt_bin), [0])
    
    row = {
        'source_bin': src_bin,
        'target_bin': tgt_bin,
        'pathway_count_mean': np.mean(counts),
        'pathway_count_std': np.std(counts),
        'pathway_count_median': np.median(counts),
        'pathway_count_q25': np.percentile(counts, 25),
        'pathway_count_q75': np.percentile(counts, 75),
        'n_pairs_in_bin': len(counts)
    }
    
    # Add signature features
    for i, val in enumerate(sig_features):
        row[f'inter_sig_{i}'] = val
    
    training_data.append(row)

df = pd.DataFrame(training_data)

print(f"Training dataset: {df.shape}")
print(f"\nColumns: {list(df.columns[:10])}... (+ {len(df.columns)-10} more)")
print(f"\nSample statistics:")
print(df[['source_bin', 'target_bin', 'pathway_count_mean', 'pathway_count_std', 'n_pairs_in_bin']].describe())

## Save Training Data

In [ ]:
output_file = output_dir / f'{metapath}_training_data.csv'
df.to_csv(output_file, index=False)

print(f"✓ Saved training data: {output_file}")
print(f"  Rows: {len(df):,}")
print(f"  Columns: {len(df.columns)}")
print(f"  File size: {output_file.stat().st_size / 1024:.1f} KB")

print(f"\n{'='*70}")
print(f"DATA PREPARATION COMPLETE: {metapath}")
print(f"{'='*70}")